# Generate Descriptors
Compute per-ligand descriptors for the CN/NN project.
- Reads `input/ML-CN-Smiles.csv` and `input/ML-NN-Smiles.csv` (`id,smiles`).
- Loads `{id}_conformers.sdf` from `Generate_LoQI/output/cn-loqi-conformers` and `nn-loqi-conformers`, as written by `Generate_LoQI/generate_loqi_generic.py`.
- Writes temporary XYZ files and computes RDKit, Morfeus/xTB, and Mordred descriptors.
- Merges `input/activity_score.csv`: `reduction_activity_score_1_fast_0_slow` becomes `activity_score`, the target used by `ML_pipeline.ipynb`.

RDKit is required to process SDF files. Morfeus/xTB and Mordred are optional.
Atom-specific descriptors require explicit **1-based SDF/XYZ atom indices**; the input CSVs do not define binding atoms. Solid/cone and bite angles require a full metal complex, which the ligand conformer files do not contain.

## User Options

In [ ]:
from pathlib import Path

# Run from IR-Ligand-ML or its parent ML directory.
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "IR-Ligand-ML":
    BASE_DIR = BASE_DIR / "IR-Ligand-ML"
IN_DIR = BASE_DIR / "input"
OUT_DIR = BASE_DIR / "output"

CN_CSV = IN_DIR / "ML-CN-Smiles.csv"
NN_CSV = IN_DIR / "ML-NN-Smiles.csv"
TARGET_CSV = IN_DIR / "activity_score.csv"

CN_CONFORMERS_DIR = BASE_DIR / "Generate_LoQI" / "output" / "cn-loqi-conformers"
NN_CONFORMERS_DIR = BASE_DIR / "Generate_LoQI" / "output" / "nn-loqi-conformers"

OUT_LIGAND_FILE = OUT_DIR / "CN_NN_ligand_descriptors.csv"
OUT_DESCRIPTORS_FILE = OUT_DIR / "CN_NN_combinations_descriptors.csv"
OUTPUT_CSV = OUT_DIR / "CN_NN_combinations_descriptors_with_score.csv"
# RDKit graph-validated SDF donor indices; see donor_atoms.py.
# C^N: cyclometalating aryl C-H. N^N: a selected pyridine-like N.
# NN_19 is fluorenone (no N); retain donor-dependent NaNs for correction.
CN_ATOM_INDICES = {
    "CN_1": {"center_idx": 3},
    "CN_2": {"center_idx": 7},
    "CN_3": {"center_idx": 7},
    "CN_4": {"center_idx": 7},
    "CN_5": {"center_idx": 7},
    "CN_6": {"center_idx": 7},
    "CN_7": {"center_idx": 7},
    "CN_8": {"center_idx": 7},
    "CN_9": {"center_idx": 5},
    "CN_10": {"center_idx": 7},
    "CN_11": {"center_idx": 7},
    "CN_12": {"center_idx": 7},
    "CN_13": {"center_idx": 7},
    "CN_14": {"center_idx": 4},
    "CN_15": {"center_idx": 5},
    "CN_23": {"center_idx": 9},
    "CN_26": {"center_idx": 3},
    "CN_27": {"center_idx": 3},
    "CN_29": {"center_idx": 7},
    "CN_30": {"center_idx": 3},
    "CN_32": {"center_idx": 4},
    "CN_33": {"center_idx": 4},
    "CN_35": {"center_idx": 5},
    "CN_36": {"center_idx": 4},
}

NN_ATOM_INDICES = {
    "NN_1": {"center_idx": 10},
    "NN_2": {"center_idx": 5},
    "NN_3": {"center_idx": 12},
    "NN_4": {"center_idx": 13},
    "NN_5": {"center_idx": 8},
    "NN_6": {"center_idx": 8},
    "NN_7": {"center_idx": 18},
    "NN_8": {"center_idx": 6},
    "NN_9": {"center_idx": 5},
    "NN_11": {"center_idx": 7},
    "NN_14": {"center_idx": 9},
    "NN_16": {"center_idx": 3},
    "NN_17": {"center_idx": 10},
    "NN_19": {"center_idx": None},
    "NN_20": {"center_idx": 4},
    "NN_21": {"center_idx": 5},
    "NN_23": {"center_idx": 25},
    "NN_24": {"center_idx": 8},
    "NN_26": {"center_idx": 8},
    "NN_33": {"center_idx": 5},
    "NN_34": {"center_idx": 13},
    "NN_37": {"center_idx": 13},
    "NN_40": {"center_idx": 10},
    "NN_44": {"center_idx": 14},
}

# AIMNet2 single-point charge inference; no geometry optimisation.
AIMNET2_MODEL = "aimnet2"
AIMNET2_DEVICE = "cpu"


## Imports and Configuration

In [ ]:
import json
import shutil
import pandas as pd
import numpy as np

import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from Generate_Descriptors.descriptor_audit import (
    DESCRIPTOR_FAILURES, record_descriptor_failure, column_missingness,
    write_descriptor_audit,
)

# RDKit availability and common symbols
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Lipinski
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

# Morfeus / xTB availability
try:
    from morfeus import read_xyz
    MORFEUS_AVAILABLE = True
except ImportError:
    MORFEUS_AVAILABLE = False

try:
    from morfeus import XTB  # optional
    XTB_AVAILABLE = True
except ImportError:
    XTB_AVAILABLE = False

## Mordred
try:
    from mordred import Calculator
    from mordred import (
        EState,
        BCUT,
        TopologicalCharge,
        Polarizability,
        Autocorrelation,
        Chi,
        BalabanJ,
        WienerIndex,
        ZagrebIndex,
        KappaShapeIndex,
        McGowanVolume,
        VdwVolumeABC,
    )
    from mordred.error import Error, Missing
    MORDRED_AVAILABLE = True
except ImportError:
    MORDRED_AVAILABLE = False

# Importing the API alone does not mean the external xtb executable is usable.
XTB_AVAILABLE = XTB_AVAILABLE and shutil.which("xtb") is not None

print('RDKIT_AVAILABLE =', RDKIT_AVAILABLE)
print('MORFEUS_AVAILABLE =', MORFEUS_AVAILABLE)
print('XTB_AVAILABLE =', XTB_AVAILABLE)

print('MORDRED_AVAILABLE =', MORDRED_AVAILABLE)


In [3]:
def find_conformer_sdf(ligand_id, mol_type, conformers_dir):
    """Resolve CN_1/NN_1 to LoQI's exact filename 1_conformers.sdf."""
    if mol_type not in {"CN", "NN"}:
        raise ValueError(f"Unknown mol_type: {mol_type}")
    ligand_id = str(ligand_id)
    prefix = f"{mol_type}_"
    if not ligand_id.startswith(prefix) or not ligand_id[len(prefix):].isdigit():
        raise ValueError(f"Expected {mol_type}_<id>, got {ligand_id!r}")
    sdf_path = Path(conformers_dir) / f"{ligand_id[len(prefix):]}_conformers.sdf"
    return sdf_path if sdf_path.is_file() else None


def read_sdf_pick_lowest_energy(sdf_path):
    """Select by explicit SDF energy properties; otherwise use the first conformer.

    LoQI's current local SDF files have no energy properties. Coordinates,
    atom counts, IDs, and other numeric metadata are not conformer energies.
    """
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required to read ligand conformers.")
    sdf_path = Path(sdf_path)
    if not sdf_path.is_file():
        raise FileNotFoundError(sdf_path)
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
    mol_energy_pairs = []
    for mol in supplier:
        if mol is None or mol.GetNumConformers() == 0:
            continue
        energy = None
        for key in mol.GetPropNames():
            if "energy" in key.lower():
                try:
                    value = float(mol.GetProp(key))
                except ValueError:
                    continue
                if np.isfinite(value):
                    energy = value
                    break
        mol_energy_pairs.append((mol, energy))
    if not mol_energy_pairs:
        return None, None
    with_energy = [(mol, energy) for mol, energy in mol_energy_pairs if energy is not None]
    return min(with_energy, key=lambda pair: pair[1]) if with_energy else mol_energy_pairs[0]


def write_xyz(mol, out_path, energy=None):
    """Write XYZ in SDF atom order, preserving 1-based descriptor indices."""
    if mol is None or mol.GetNumConformers() == 0:
        raise ValueError("An RDKit molecule with a conformer is required.")
    conf = mol.GetConformer()
    lines = [str(mol.GetNumAtoms()),
             f"Energy {energy}" if energy is not None else "Converted from LoQI SDF"]
    for i, atom in enumerate(mol.GetAtoms()):
        pos = conf.GetAtomPosition(i)
        lines.append(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(lines) + "\n")
    return str(out_path)


## RDKit
| Descriptor column   | Package/source                             |
| ------------------- | ------------------------------------------ |
| `MolWt`             | RDKit `Descriptors.MolWt`                  |
| `LogP`              | RDKit `Crippen.MolLogP`                    |
| `TPSA`              | RDKit `rdMolDescriptors.CalcTPSA`          |
| `NumHDonors`        | RDKit `Lipinski.NumHDonors`                |
| `NumHAcceptors`     | RDKit `Lipinski.NumHAcceptors`             |
| `NumRotatableBonds` | RDKit `Lipinski.NumRotatableBonds`         |
| `NumHeavyAtoms`     | RDKit `rdMolDescriptors.CalcNumHeavyAtoms` |


In [4]:
# RDKit descriptor functions
def compute_rdkit_descriptors(mol):
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required to compute ligand descriptors.")
    if mol is None:
        return {}
    try:
        desc = {}
        desc['MolWt'] = float(Descriptors.MolWt(mol))
        desc['LogP'] = float(Crippen.MolLogP(mol))
        desc['TPSA'] = float(rdMolDescriptors.CalcTPSA(mol))
        desc['NumHDonors'] = int(Lipinski.NumHDonors(mol))
        desc['NumHAcceptors'] = int(Lipinski.NumHAcceptors(mol))
        desc['NumRotatableBonds'] = int(Lipinski.NumRotatableBonds(mol))
        desc['NumHeavyAtoms'] = int(rdMolDescriptors.CalcNumHeavyAtoms(mol))
        return desc
    except Exception as e:
        return {'rdkit_error': str(e)}


print('RDKit descriptor functions ready')


RDKit descriptor functions ready


## Morfeus Descriptors
| Descriptor column                 | Package/source                                                            |
| --------------------------------- | ------------------------------------------------------------------------- |
| `Morfeus_Pint`                    | Morfeus `Dispersion`, from `disp.p_int`                                   |
| `Morfeus_DispersionArea`          | Morfeus `Dispersion`, from `disp.area`                                    |
| `Morfeus_DispersionVol`           | Morfeus `Dispersion`, from `disp.volume`                                  |
| `Morfeus_SASA`                    | Morfeus `SASA`, from `sasa_obj.area`                                      |
| `Morfeus_SASAVolume`              | Morfeus `SASA`, from `sasa_obj.volume`                                    |
| `Morfeus_SASA_atom_areas`         | Morfeus `SASA`, from `sasa_obj.atom_areas`                                |
| `Morfeus_BuriedVolume_Calpha`     | Morfeus `BuriedVolume`; CN ligand reaction center                         |
| `Morfeus_BuriedVolume_N`          | Morfeus `BuriedVolume`; NN ligand reaction center                         |
| `Morfeus_Sterimol_L`              | Morfeus `Sterimol`, from `s.L_value`                                      |
| `Morfeus_Sterimol_B1`             | Morfeus `Sterimol`, from `s.B_1_value`                                    |
| `Morfeus_Sterimol_B5`             | Morfeus `Sterimol`, from `s.B_5_value`                                    |

| `Morfeus_DummyCs_BuriedVolume_pct` | Morfeus `BuriedVolume`, centered on dummy Cs, percent |
| `Morfeus_DummyCs_SolidAngle_steradians` | Morfeus `SolidAngle.solid_angle`, centered on dummy Cs, steradians |
| `Morfeus_DummyCs_SolidConeAngle_deg` | Morfeus `SolidAngle.cone_angle`, centered on dummy Cs, degrees |


The three dummy-Cs descriptors use one selected donor (`center_idx`) and a virtual Cs center placed 2.5 Å outward from its heavy-atom neighbors. Inspect this geometric approximation on representative ligands. `CN_ATOM_INDICES` and `NN_ATOM_INDICES` in User Options contain graph-validated selections (except NN_19, flagged for correction); `dummy_idx` remains a ligand atom. Cs is never added to the SDF, XYZ, or xTB calculation. Bite angles are skipped by the ligand runner.

[Morfeus buried-volume documentation](https://digital-chemistry-laboratory.github.io/morfeus/buried_volume.html) specifies a 1-based central atom index and excludes the central atom from occupied volume.

[Morfeus solid-angle documentation](https://digital-chemistry-laboratory.github.io/morfeus/solid_angle.html) describes `solid_angle` (steradians) and the equivalent `cone_angle` (degrees). Both use the same dummy-Cs geometry; no separate `ConeAngle` or bite-angle calculation is used in the ligand pipeline.


In [5]:
def _read_morfeus_geometry(xyz_file):
    if not MORFEUS_AVAILABLE:
        raise ImportError("Morfeus is required to read XYZ geometry.")
    elements, coords = read_xyz(str(xyz_file))
    from morfeus.utils import convert_elements
    elements = convert_elements(elements, output="symbols")
    if hasattr(coords, 'ndim') and coords.ndim == 3:
        coords = coords[0]
    return list(elements), np.asarray(coords)


def find_reaction_center(elements, coordinates, mol_type, center_idx=None, dummy_idx=None):
    """Validate supplied 1-based atom indices; do not infer binding sites."""
    if mol_type not in {"CN", "NN"}:
        raise ValueError(f"Unknown mol_type: {mol_type}")
    if np.asarray(coordinates).shape != (len(elements), 3):
        raise ValueError("Coordinates must have shape (number of atoms, 3).")
    for index in (center_idx, dummy_idx):
        if index is not None and (
            not isinstance(index, (int, np.integer)) or not 1 <= index <= len(elements)
        ):
            raise ValueError(f"Atom index must be between 1 and {len(elements)}: {index}")
    if center_idx is not None and center_idx == dummy_idx:
        raise ValueError("Center and dummy atoms must be different.")
    return {"center_idx": center_idx, "dummy_idx": dummy_idx}


# Ligand-only Morfeus descriptors: Dispersion (P_int), SASA, Buried Volume, Sterimol

def morfeus_dispersion(xyz_file):
    try:
        from morfeus import Dispersion
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    try:
        disp = Dispersion(elements, coords)
        return {
            'Morfeus_Pint': float(disp.p_int),
            'Morfeus_DispersionArea': float(disp.area),
            'Morfeus_DispersionVol': float(disp.volume),
        }
    except Exception:
        return {}


def morfeus_sasa(xyz_file):
    try:
        from morfeus import SASA
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    try:
        sasa_obj = SASA(elements, coords)
        out = {'Morfeus_SASA': float(sasa_obj.area), 'Morfeus_SASAVolume': float(sasa_obj.volume)}
        # per-atom areas available as sasa_obj.atom_areas (1-indexed keys)
        try:
            out['Morfeus_SASA_atom_areas'] = dict(sasa_obj.atom_areas)
        except Exception:
            pass
        return out
    except Exception:
        return {}


def morfeus_buried_volume(xyz_file, center_idx, mol_type):
    try:
        from morfeus import BuriedVolume
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    centers = find_reaction_center(elements, coords, mol_type, center_idx=center_idx)
    center_idx = centers['center_idx']
    if not center_idx:
        return {}
    try:
        bv = BuriedVolume(elements, coords, center_idx)
        label = 'Calpha' if mol_type == 'CN' else 'N'
        return {f'Morfeus_BuriedVolume_{label}': float(bv.fraction_buried_volume * 100)}
    except Exception:
        return {}


def morfeus_sterimol(xyz_file, center_idx, dummy_idx, mol_type):
    try:
        from morfeus import Sterimol
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    centers = find_reaction_center(elements, coords, mol_type, center_idx, dummy_idx)
    center_idx, dummy_idx = centers['center_idx'], centers['dummy_idx']
    if not center_idx or not dummy_idx:
        return {}
    try:
        s = Sterimol(elements, coords, dummy_idx, center_idx)
        return {'Morfeus_Sterimol_L': float(s.L_value), 'Morfeus_Sterimol_B1': float(s.B_1_value), 'Morfeus_Sterimol_B5': float(s.B_5_value)}
    except Exception:
        return {}

def morfeus_dummy_cs(mol, donor_idx, distance=2.5):
    """Measure buried volume, solid angle, and solid-cone angle around dummy Cs.

    donor_idx is a 1-based index in the original SDF molecule. The outward
    direction is determined from the donor's heavy-atom neighbors. The 2.5 Å
    separation is an assumed geometric parameter, not a validated bond length.
    Cs is added only to a copied Morfeus geometry; the RDKit mol is unchanged.
    """
    from morfeus import BuriedVolume, SolidAngle

    if mol is None or mol.GetNumConformers() == 0:
        raise ValueError("A ligand molecule with a conformer is required.")
    if not isinstance(donor_idx, (int, np.integer)) or not 1 <= donor_idx <= mol.GetNumAtoms():
        raise ValueError("donor_idx must be a valid 1-based ligand atom index.")
    if not np.isfinite(distance) or distance <= 0:
        raise ValueError("Dummy-Cs distance must be finite and positive.")

    coords = np.array(mol.GetConformer().GetPositions(), dtype=float, copy=True)
    if not np.isfinite(coords).all():
        raise ValueError("Ligand coordinates must be finite.")
    elements = [atom.GetSymbol() for atom in mol.GetAtoms()]
    i = donor_idx - 1
    donor = mol.GetAtomWithIdx(i)
    neighbors = [atom.GetIdx() for atom in donor.GetNeighbors() if atom.GetAtomicNum() > 1]
    if not neighbors:
        raise ValueError("Donor has no heavy-atom neighbors.")
    direction = coords[i] - coords[neighbors].mean(axis=0)
    norm = np.linalg.norm(direction)
    if norm < 1e-8:
        raise ValueError("Cannot determine dummy-metal direction.")
    cs_position = coords[i] + distance * direction / norm
    elements.append("Cs")
    coords = np.vstack([coords, cs_position])
    cs_index = len(elements)
    bv = BuriedVolume(elements, coords, cs_index)
    sa = SolidAngle(elements, coords, cs_index)
    return {
        "Morfeus_DummyCs_BuriedVolume_pct": 100 * float(bv.fraction_buried_volume),
        "Morfeus_DummyCs_SolidAngle_steradians": float(sa.solid_angle),
        "Morfeus_DummyCs_SolidConeAngle_deg": float(sa.cone_angle),
    }


### Unused, complex-specific descriptors

These retained functions require a full metal complex. They are not called by the current single-donor dummy-Cs ligand pipeline.

| Descriptor column | Package/source |
| ----------------- | -------------- |
| `Morfeus_SolidAngle_steradians`*   | Morfeus `SolidAngle`, from `sa.solid_angle`                               |
| `Morfeus_SolidAngle_cone_deg`  *   | Morfeus `SolidAngle`, from `sa.cone_angle`                                |
| `Morfeus_SolidAngle_G` *           | Morfeus `SolidAngle`, from `sa.G`                                         |
| `Morfeus_ConeAngle_deg` *          | Morfeus `ConeAngle`, from `ca.cone_angle`                                 |
| `Morfeus_ConeAngle_tangent_atoms*`* | Morfeus `ConeAngle`, from `ca.tangent_atoms`                              |
| `Morfeus_BiteAngle_deg` *          | Morfeus `BiteAngle`, from `ba.angle`                                      |


In [6]:
# Complex-specific Morfeus descriptors: Solid/Cone Angle, Bite Angle
# Keep these functions for now, but not called for ligand-only XYZ files.

def morfeus_solid_and_cone(xyz_file, metal_index):
    """
    Compute solid angle and cone angle.

    Only use this for full Ir complexes or structures with a real metal center. Not for ligand-only XYZ files.
    """
    try:
        from morfeus import SolidAngle, ConeAngle
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    if not 1 <= metal_index <= len(elements) or elements[metal_index - 1] != 'Ir':
        raise ValueError("metal_index must identify the Ir atom in a full complex.")
    out = {}
    try:
        sa = SolidAngle(elements, coords, metal_index)
        out['Morfeus_SolidAngle_steradians'] = float(sa.solid_angle)
        out['Morfeus_SolidAngle_cone_deg'] = float(sa.cone_angle)
        out['Morfeus_SolidAngle_G'] = float(sa.G)
    except Exception:
        pass
    try:
        ca = ConeAngle(elements, coords, metal_index)
        out['Morfeus_ConeAngle_deg'] = float(ca.cone_angle)
        try:
            out['Morfeus_ConeAngle_tangent_atoms'] = list(ca.tangent_atoms)
        except Exception:
            pass
    except Exception:
        pass
    return out


def morfeus_bite_angle(xyz_file, metal_index, ligand_idx1, ligand_idx2, ref_atoms=None):
    try:
        from morfeus import BiteAngle
    except ImportError:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    if not 1 <= metal_index <= len(elements) or elements[metal_index - 1] != 'Ir':
        raise ValueError("metal_index must identify the Ir atom in a full complex.")
    if len({metal_index, ligand_idx1, ligand_idx2}) != 3 or any(
        not 1 <= index <= len(elements) for index in (ligand_idx1, ligand_idx2)
    ):
        raise ValueError("Provide two distinct donor atom indices and the Ir index.")
    try:
        ba = BiteAngle(coords, metal_index, ligand_idx1, ligand_idx2, ref_atoms=ref_atoms)
        return {'Morfeus_BiteAngle_deg': float(ba.angle), 'Morfeus_BiteAngle_inverted': bool(ba.inverted)}
    except Exception as e:
        return {'morfeus_bite_angle_error': str(e)}


| Descriptor column        | Package/source                                          |
| ------------------------ | ------------------------------------------------------- |
| `xTB_HOMO`               | Morfeus `XTB.get_homo()`                                |
| `xTB_LUMO`               | Morfeus `XTB.get_lumo()`                                |
| `xTB_HOMO_LUMO_gap`      | computed from `LUMO - HOMO`                             |
| `xTB_IP`                 | Morfeus `XTB.get_ip(corrected=True)`                    |
| `xTB_EA`                 | Morfeus `XTB.get_ea(corrected=True)`                    |
| `xTB_Electrophilicity`   | Morfeus `XTB.get_global_descriptor("electrophilicity")` |
| `xTB_Nucleophilicity`    | Morfeus `XTB.get_global_descriptor("nucleophilicity")`  |
| `xTB_Electrofugality`    | Morfeus `XTB.get_global_descriptor("electrofugality")`  |
| `xTB_Nucleofugality`     | Morfeus `XTB.get_global_descriptor("nucleofugality")`   |
| `xTB_Chemical_Potential` | computed from `-(IP + EA) / 2`                          |
| `xTB_Hardness`           | computed from `(IP - EA) / 2`                           |
| `xTB_Softness`           | computed from `1 / (2 * hardness)`                      |
| `xTB_Dipole`             | Morfeus `XTB.get_dipole()`, then vector norm            |
| `xTB_Charge_Calpha`      | Morfeus `XTB.get_charges()`; CN reaction center         |
| `xTB_Charge_N`           | Morfeus `XTB.get_charges()`; NN reaction center         |
| `xTB_Charge_Br`          | Morfeus `XTB.get_charges()`; CN Br dummy/leaving atom   |
| `xTB_Fukui_plus_Calpha`  | Morfeus `XTB.get_fukui("electrophilicity")`; CN center  |
| `xTB_Fukui_plus_N`       | Morfeus `XTB.get_fukui("electrophilicity")`; NN center  |
| `xTB_Fukui_minus_Calpha` | Morfeus `XTB.get_fukui("nucleophilicity")`; CN center   |
| `xTB_Fukui_minus_N`      | Morfeus `XTB.get_fukui("nucleophilicity")`; NN center   |
| `xTB_Fukui_dual_Calpha`  | Morfeus `XTB.get_fukui("dual")`; CN center              |
| `xTB_Fukui_dual_N`       | Morfeus `XTB.get_fukui("dual")`; NN center              |


In [7]:
# Extended xTB descriptors via morfeus.XTB

def morfeus_xtb_descriptors(xyz_file, mol_type, center_idx=None, dummy_idx=None):
    if not XTB_AVAILABLE:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    xtb = XTB(elements, coords)
    desc = {}
    try:
        homo = xtb.get_homo()
        lumo = xtb.get_lumo()
        desc['xTB_HOMO'] = float(homo)
        desc['xTB_LUMO'] = float(lumo)
        desc['xTB_HOMO_LUMO_gap'] = float(lumo - homo)
    except Exception:
        pass
    try:
        desc['xTB_IP'] = float(xtb.get_ip(corrected=True))
        desc['xTB_EA'] = float(xtb.get_ea(corrected=True))
    except Exception:
        pass
    for variety, col in [
        ('electrophilicity', 'xTB_Electrophilicity'),
        ('nucleophilicity', 'xTB_Nucleophilicity'),
        ('electrofugality', 'xTB_Electrofugality'),
        ('nucleofugality', 'xTB_Nucleofugality'),
    ]:
        try:
            val = xtb.get_global_descriptor(variety)
            desc[col] = float(val)
        except Exception:
            pass
    try:
        ip = desc.get('xTB_IP')
        ea = desc.get('xTB_EA')
        if ip is not None and ea is not None:
            desc['xTB_Chemical_Potential'] = float(-(ip + ea) / 2.0)
            desc['xTB_Hardness'] = float((ip - ea) / 2.0)
            eta = desc.get('xTB_Hardness')
            if eta and eta > 0:
                desc['xTB_Softness'] = float(1.0 / (2.0 * eta))
    except Exception:
        pass
    try:
        dip = xtb.get_dipole()
        desc['xTB_Dipole'] = float(np.linalg.norm(dip)) if hasattr(dip, '__len__') else float(dip)
    except Exception:
        pass
    # Morfeus XTB charges/Fukui values use dictionaries with 1-based keys.
    centers = find_reaction_center(elements, coords, mol_type, center_idx, dummy_idx)
    center_idx, dummy_idx = centers['center_idx'], centers['dummy_idx']
    label = "Calpha" if mol_type == "CN" else "N"
    if center_idx is not None:
        try:
            charges = xtb.get_charges()
            desc[f'xTB_Charge_{label}'] = float(charges[center_idx])
        except Exception as e:
            desc['xtb_charges_error'] = str(e)
        for variety, suffix in [('electrophilicity', 'plus'), ('nucleophilicity', 'minus'), ('dual', 'dual')]:
            try:
                fukui = xtb.get_fukui(variety)
                desc[f'xTB_Fukui_{suffix}_{label}'] = float(fukui[center_idx])
            except Exception as e:
                desc[f'xtb_fukui_{suffix}_error'] = str(e)
    if dummy_idx is not None and mol_type == 'CN' and elements[dummy_idx - 1] == 'Br':
        try:
            desc['xTB_Charge_Br'] = float(xtb.get_charges()[dummy_idx])
        except Exception as e:
            desc['xtb_charge_br_error'] = str(e)
    return desc

print('Appended Morfeus/xTB advanced feature functions')


Appended Morfeus/xTB advanced feature functions


In [8]:
def compute_morfeus_descriptors(xyz_file, mol_type, run_xtb=True, center_idx=None, dummy_idx=None):
    """Compute free-ligand descriptors with optional explicit 1-based atom indices.

    Local ligand files have no Ir atom; complex-only angles are callable separately.
    Center-specific values are calculated only when their atom indices are supplied.
    """
    if xyz_file is None or not MORFEUS_AVAILABLE:
        return {}
    results = {}
    modules = [("dispersion", morfeus_dispersion, {}), ("sasa", morfeus_sasa, {})]
    if center_idx is not None:
        modules.append(("buried_volume", morfeus_buried_volume,
                        {"mol_type": mol_type, "center_idx": center_idx}))
        if dummy_idx is not None:
            modules.append(("sterimol", morfeus_sterimol,
                            {"mol_type": mol_type, "center_idx": center_idx, "dummy_idx": dummy_idx}))
    if run_xtb and XTB_AVAILABLE:
        modules.append(("xtb", morfeus_xtb_descriptors,
                        {"mol_type": mol_type, "center_idx": center_idx, "dummy_idx": dummy_idx}))
    for name, func, kwargs in modules:
        try:
            results.update(func(xyz_file, **kwargs))
        except Exception as e:
            results[f"morfeus_{name}_error"] = str(e)
    return results


## Mordred
| Descriptor column             | Package/source                      |
| ----------------------------- | ----------------------------------- |
| `Mordred_EState_*`            | Mordred `mordred.EState`            |
| `Mordred_BCUT_*`              | Mordred `mordred.BCUT`              |
| `Mordred_TopologicalCharge_*` | Mordred `mordred.TopologicalCharge` |
| `Mordred_Polarizability_*`    | Mordred `mordred.Polarizability`    |
| `Mordred_Autocorrelation_*`   | Mordred `mordred.Autocorrelation`   |
| `Mordred_Chi_*`               | Mordred `mordred.Chi`               |
| `Mordred_BalabanJ_*`          | Mordred `mordred.BalabanJ`          |
| `Mordred_WienerIndex_*`       | Mordred `mordred.WienerIndex`       |
| `Mordred_ZagrebIndex_*`       | Mordred `mordred.ZagrebIndex`       |
| `Mordred_KappaShapeIndex_*`   | Mordred `mordred.KappaShapeIndex`   |
| `Mordred_McGowanVolume`       | Mordred `mordred.McGowanVolume`     |
| `Mordred_VdwVolumeABC`        | Mordred `mordred.VdwVolumeABC`      |

Undefined Mordred values are logged in `output/descriptor_audit/descriptor_failures.csv`. EState minima/maxima for absent atom types remain NaN; `Vabc` is unsupported for iodine-containing CN_6. Universally undefined descriptor definitions are documented in `mordred_applicability_audit.csv` and are not added as fabricated columns.


In [ ]:
# Mordred descriptor functions (selected)
def compute_mordred_descriptors(mol, ligand_id=None):
    if mol is None or not MORDRED_AVAILABLE:
        return {}

    try:
        desc = {}

        selected_mordred_modules = [
            EState,
            BCUT,
            TopologicalCharge,
            Polarizability,
            Autocorrelation,
            Chi,
            BalabanJ,
            WienerIndex,
            ZagrebIndex,
            KappaShapeIndex,
            McGowanVolume,
            VdwVolumeABC,
        ]

        calc = Calculator(selected_mordred_modules, ignore_3D=True)
        result = calc(mol)

        for key, value in result.asdict().items():
            col = f"Mordred_{key}"

            # Skip Mordred calculation errors/missing values
            if isinstance(value, (Error, Missing)):
                reason = str(value)
                if str(key).startswith(("MAX", "MIN")) and "empty" in reason:
                    category = "inapplicable_absent_atom_type"
                elif str(key) == "Vabc" and "unknown atom type" in reason:
                    category = "unsupported_element"
                else:
                    category = "calculation_failure"
                record_descriptor_failure(ligand_id, col, reason, category)
                continue

            # Keep only numeric values
            try:
                value = float(value)
                if np.isfinite(value):
                    desc[col] = value
                else:
                    record_descriptor_failure(ligand_id, col, "Non-finite result", "nonfinite_result")
            except Exception as e:
                record_descriptor_failure(ligand_id, col, str(e))
                continue

        return desc

    except Exception as e:
        record_descriptor_failure(ligand_id, "Mordred", str(e))
        return {"mordred_error": str(e)}

print("Mordred descriptor functions ready")


## AIMNet2 Electronic Descriptors

The separate module `Generate_Descriptors/aimnet2_descriptors.py` reuses the selected LoQI SDF conformer, including its explicit hydrogen atoms, atom order, and RDKit formal molecular charge. It follows the input schema used in `Generate_LoQI/LoQI/src/megalodon/metrics/molecule_metrics_aimnet2.py`, without that module's optimisation or force calculations. Install the optional dependencies from `Generate_Descriptors/requirements-aimnet2.txt` in the notebook kernel (for example, `%pip install -r Generate_Descriptors/requirements-aimnet2.txt`). The official `aimnet` calculator downloads the configured model on first use.

| Descriptor | Definition |
| ---------- | ---------- |
| `AIMNet2_Charge_Calpha_or_N` | AIMNet2 atomic charge (e) at the 1-based `center_idx` supplied by `CN_ATOM_INDICES` / `NN_ATOM_INDICES` |
| `AIMNet2_NeighborChargeMean` | Mean charge (e) of the coordinating atom's directly bonded heavy-atom neighbors |
| `AIMNet2_Dipole_D` | Magnitude of the charge-derived molecular dipole, converted from e·Å to Debye using 4.80320471257 D/(e·Å) |

All atoms, including hydrogens, contribute to the dipole. Its origin is the ligand center of mass; this defines the convention for charged ligands. This is a point-charge estimate from AIMNet2 charges, not a separate learned dipole head. Dummy Cs is excluded. No geometries are regenerated or optimised. Forces, stress, and Hessians are disabled; the calculator's intrinsic energy output is discarded. No Br-specific AIMNet2 descriptors are calculated.

Each ligand retains all three numeric columns. Failures are logged with the ligand ID and remain `NaN`. Missing donor mappings affect the two local descriptors; the dipole can still be computed. Unsupported elements, incomplete hydrogens, radical ligands, or inconsistent charge sums are logged as failures. Donor mappings are now populated and graph-validated in Main. NN_19 remains an explicit no-N exception pending input correction.

[AIMNet2 calculator documentation](https://github.com/isayevlab/aimnetcentral) specifies coordinates in Å and atomic charges in e.


In [ ]:
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from Generate_Descriptors.aimnet2_descriptors import (
    AIMNET2_COLUMNS,
    LOGGER as AIMNET2_LOGGER,
    compute_aimnet2_descriptors,
    empty_aimnet2_descriptors,
)

from Generate_Descriptors.donor_atoms import validate_donor_mappings

DUMMY_CS_COLUMNS = (
    "Morfeus_DummyCs_BuriedVolume_pct",
    "Morfeus_DummyCs_SolidAngle_steradians",
    "Morfeus_DummyCs_SolidConeAngle_deg",
)


## Processing Function

In [ ]:
def process_ligand(ligand_id, mol_type, conformers_dir, write_xyz_to=None,
                   run_xtb=True, center_idx=None, dummy_idx=None):
    """Load one ligand and compute its existing descriptor families."""
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required to process ligands.")
    if write_xyz_to is None:
        write_xyz_to = OUT_DIR / "tmp_smoke_xyz"
    result = {**empty_aimnet2_descriptors(), **dict.fromkeys(DUMMY_CS_COLUMNS, np.nan), "ligand_id": ligand_id, "mol_type": mol_type}
    sdf_path = find_conformer_sdf(ligand_id, mol_type, conformers_dir)
    if sdf_path is None:
        result.update(status="sdf_missing", sdf_missing=True)
        AIMNET2_LOGGER.warning("AIMNet2 skipped for %s: conformer SDF is missing.", ligand_id)
        return result
    result['sdf_file'] = str(sdf_path)
    mol, energy = read_sdf_pick_lowest_energy(sdf_path)
    if mol is None:
        result['status'] = 'no_mol'
        AIMNET2_LOGGER.warning("AIMNet2 skipped for %s: no valid SDF molecule.", ligand_id)
        return result
    result['smiles'] = Chem.MolToSmiles(Chem.RemoveHs(mol))
    xyz_path = Path(write_xyz_to) / f"{ligand_id}_conformer.xyz"
    result['xyz_file'] = write_xyz(mol, xyz_path, energy)
    result.update(compute_rdkit_descriptors(mol))
    result.update(compute_mordred_descriptors(mol, ligand_id=ligand_id))
    result.update(compute_aimnet2_descriptors(
        mol, center_idx=center_idx, ligand_id=ligand_id,
        model=AIMNET2_MODEL, device=AIMNET2_DEVICE,
    ))
    result.update(compute_morfeus_descriptors(
        result['xyz_file'], mol_type, run_xtb=run_xtb,
        center_idx=center_idx, dummy_idx=dummy_idx,
    ))
    if center_idx is None:
        reason = "NN_19 has no N donor; flagged for structure correction." if ligand_id == "NN_19" else "No validated donor index."
        for col in list(AIMNET2_COLUMNS[:2]) + list(DUMMY_CS_COLUMNS) + [f"Morfeus_BuriedVolume_{'Calpha' if mol_type == 'CN' else 'N'}"]:
            record_descriptor_failure(ligand_id, col, reason, "missing_valid_donor")
    # Cs is added only after the original-ligand descriptor calculations.
    if center_idx is not None and MORFEUS_AVAILABLE:
        try:
            result.update(morfeus_dummy_cs(mol, center_idx))
        except Exception as e:
            result['morfeus_dummy_cs_error'] = str(e)
            for col in DUMMY_CS_COLUMNS:
                record_descriptor_failure(ligand_id, col, str(e))
    for col, value in result.items():
        if col.endswith("_error"):
            record_descriptor_failure(ligand_id, col, value)
        elif col in AIMNET2_COLUMNS and pd.isna(value) and center_idx is not None:
            record_descriptor_failure(ligand_id, col, "AIMNet2 result missing; see logged inference failure.")
    result['status'] = 'ok'
    return result


def read_ligand_list(csv_path, prefix, sample_n=None):
    """Read the project's explicit id,smiles schema and return IDs and SMILES."""
    if prefix not in {"CN", "NN"}:
        raise ValueError(f"Unknown ligand prefix: {prefix}")
    df = pd.read_csv(csv_path, dtype={"id": str, "smiles": str})
    df = df[["id", "smiles"]]
    if df.isna().any().any() or df['id'].duplicated().any():
        raise ValueError("Ligand IDs must be unique, and IDs/SMILES must not be missing.")
    if not df['id'].str.fullmatch(r"\d+").all():
        raise ValueError("Ligand IDs must be numeric, as in the LoQI output filenames.")
    if sample_n is not None:
        df = df.head(sample_n)
    ids = (prefix + "_" + df['id']).tolist()
    return ids, dict(zip(ids, df['smiles']))


def _safe_process_and_strip(res):
    """Remove non-feature metadata and make list/dict values CSV-friendly."""
    if not res:
        return {}

    # Keep smiles and mol_type as metadata, but remove file/status fields.
    bad_keys = {
        "status",
        "sdf_file",
        "ligand_id",
        "xyz_file",
    }

    out = {}
    for k, v in res.items():
        if k in bad_keys:
            continue

        if isinstance(v, (list, dict)):
            try:
                out[k] = json.dumps(v)
            except Exception:
                out[k] = str(v)
        else:
            out[k] = v

    return out


In [12]:
def compute_ligand_feature_dict(ligand_ids, mol_type, conformer_dir, tmp_xyz_dir,
                                run_xtb=True, atom_indices=None):
    """Compute once per ligand; atom_indices maps IDs to center_idx/dummy_idx."""
    atom_indices = {} if atom_indices is None else atom_indices
    features = {}

    for ligand_id in ligand_ids:
        try:
            print(f"Processing {mol_type}:", ligand_id)

            res = process_ligand(
                ligand_id,
                mol_type,
                conformer_dir,
                write_xyz_to=tmp_xyz_dir,
                run_xtb=run_xtb,
                **atom_indices.get(ligand_id, {}),
            )

            features[ligand_id] = _safe_process_and_strip(res)

        except Exception as e:
            features[ligand_id] = {**empty_aimnet2_descriptors(), "error": str(e)}
            AIMNET2_LOGGER.warning("AIMNet2 skipped for %s: ligand processing failed: %s", ligand_id, e)

    return features


def build_pairwise_descriptor_table(cn_ids, nn_ids, cn_features, nn_features):
    """Build CN-NN pairwise descriptor table."""
    rows = []

    for cid in cn_ids:
        for nid in nn_ids:
            row = {
                "combo": f"{cid}#{nid}",
                "CN_ligand_id": cid,
                "NN_ligand_id": nid,
            }

            for k, v in cn_features.get(cid, {}).items():
                row[f"CN_{k}"] = v

            for k, v in nn_features.get(nid, {}).items():
                row[f"NN_{k}"] = v

            rows.append(row)

    return pd.DataFrame(rows)


def merge_activity_score(descriptor_df, target_csv, output_file=None):
    """Merge the exact target schema from input/activity_score.csv."""
    target_col = "reduction_activity_score_1_fast_0_slow"
    target_df = pd.read_csv(target_csv, usecols=["combo", target_col])
    target_df = target_df.rename(columns={target_col: "activity_score"})
    out_df = descriptor_df.merge(target_df, on="combo", how="left", validate="many_to_one")
    print("Missing activity_score values after merge:", out_df['activity_score'].isna().sum())
    if output_file is not None:
        output_file = Path(output_file)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        out_df.to_csv(output_file, index=False)
        print("Wrote descriptors + activity_score to:", output_file)
    return out_df


## Main

In [ ]:
# Run the cells above before this cell.
if not RDKIT_AVAILABLE:
    raise ImportError("RDKit is required to generate ligand descriptors.")

cn_ids, cn_smiles = read_ligand_list(CN_CSV, "CN")
nn_ids, nn_smiles = read_ligand_list(NN_CSV, "NN")

# Validate every configured donor against the actual SDF graph before computing.
DESCRIPTOR_FAILURES.clear()
donor_records = validate_donor_mappings(
    cn_ids, nn_ids, cn_smiles, nn_smiles, CN_ATOM_INDICES, NN_ATOM_INDICES,
    find_conformer_sdf, read_sdf_pick_lowest_energy,
    CN_CONFORMERS_DIR, NN_CONFORMERS_DIR,
)
audit_dir = OUT_DIR / "descriptor_audit"
audit_dir.mkdir(parents=True, exist_ok=True)
before_audit = audit_dir / "baseline_column_audit.csv"
if OUTPUT_CSV.exists() and not before_audit.exists():
    column_missingness(pd.read_csv(OUTPUT_CSV)).to_csv(before_audit, index=False)
if not XTB_AVAILABLE:
    record_descriptor_failure("__all__", "xTB", "xTB API or executable is unavailable; no xTB values were fabricated.", "unavailable_dependency")

OUT_DIR.mkdir(parents=True, exist_ok=True)
tmp_xyz_dir = OUT_DIR / "tmp_xyz"

cn_features = compute_ligand_feature_dict(
    cn_ids, "CN", CN_CONFORMERS_DIR, tmp_xyz_dir, run_xtb=XTB_AVAILABLE,
    atom_indices=CN_ATOM_INDICES,
)
nn_features = compute_ligand_feature_dict(
    nn_ids, "NN", NN_CONFORMERS_DIR, tmp_xyz_dir, run_xtb=XTB_AVAILABLE,
    atom_indices=NN_ATOM_INDICES,
)

# Retain input SMILES even when an individual conformer could not be processed.
for ligand_id in cn_ids:
    cn_features[ligand_id].setdefault("smiles", cn_smiles[ligand_id])
for ligand_id in nn_ids:
    nn_features[ligand_id].setdefault("smiles", nn_smiles[ligand_id])

ligand_df = pd.DataFrame(
    [{**cn_features[ligand_id], "ligand_id": ligand_id, "mol_type": "CN"}
     for ligand_id in cn_ids]
    + [{**nn_features[ligand_id], "ligand_id": ligand_id, "mol_type": "NN"}
       for ligand_id in nn_ids]
)
ligand_df.to_csv(OUT_LIGAND_FILE, index=False)

# Reuse each ligand's descriptors across all CN-NN combinations.
descriptor_df = build_pairwise_descriptor_table(cn_ids, nn_ids, cn_features, nn_features)
descriptor_df.to_csv(OUT_DESCRIPTORS_FILE, index=False)
scored_descriptor_df = merge_activity_score(descriptor_df, TARGET_CSV, OUTPUT_CSV)

print(f"Saved {len(ligand_df)} ligands to {OUT_LIGAND_FILE}")
print(f"Saved {len(descriptor_df)} combinations to {OUT_DESCRIPTORS_FILE}")
print(f"Scored descriptor table shape: {scored_descriptor_df.shape}")
print("Missing AIMNet2 values per ligand column:")
print(ligand_df[list(AIMNET2_COLUMNS)].isna().sum())
output_audit = write_descriptor_audit(ligand_df, scored_descriptor_df, donor_records, audit_dir)
print(f"Audited all {len(output_audit)} output columns; {(output_audit['missing_count'] > 0).sum()} retain documented missing values.")
print("Detailed audit:", audit_dir)
display(scored_descriptor_df.head())
